# Step 2: Spark SQL & Catalyst Optimizer

## Learning Objectives
1. Spark SQL deep dive (complex queries, subqueries, CTEs)
2. How the Catalyst Optimizer works
3. How to read Execution Plans
4. Key optimization techniques: Predicate Pushdown, Column Pruning, Constant Folding
5. DataFrame API vs SQL execution plan comparison

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import random

spark = SparkSession.builder \
    .appName("Step2-SparkSQL-Catalyst") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"✅ Spark UI: http://localhost:4040")

Spark version: 3.5.0
✅ Spark UI: http://localhost:4040


---
## 1. Generating Practice Data

Generate online shopping mall data: customers, orders, and products tables.

In [2]:
random.seed(42)

# Customers table
regions = ["Seoul", "Gyeonggi", "Busan", "Daegu", "Incheon", "Gwangju", "Daejeon", "Jeju"]
tiers = ["Bronze", "Silver", "Gold", "Platinum"]

customers = [
    (i, f"customer_{i}", random.choice(regions), random.choice(tiers), random.randint(20, 60))
    for i in range(1, 10001)
]
customers_df = spark.createDataFrame(customers, ["customer_id", "name", "region", "tier", "age"])

# Products table
categories = ["Electronics", "Apparel", "Food", "Books", "Sports", "Furniture"]
products = [
    (i, f"product_{i}", random.choice(categories), random.randint(1000, 500000))
    for i in range(1, 501)
]
products_df = spark.createDataFrame(products, ["product_id", "product_name", "category", "price"])

# Orders table (500K rows)
statuses = ["completed", "completed", "completed", "completed", "cancelled", "returned"]  # weighted toward completed
orders = [
    (i, random.randint(1, 10000), random.randint(1, 500),
     random.randint(1, 10), random.choice(statuses),
     f"2025-{random.randint(1,12):02d}-{random.randint(1,28):02d}")
    for i in range(1, 500001)
]
orders_df = spark.createDataFrame(orders, ["order_id", "customer_id", "product_id", "quantity", "status", "order_date"])

# Register as temp views
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")
orders_df.createOrReplaceTempView("orders")

print(f"Customers: {customers_df.count():,}")
print(f"Products:  {products_df.count():,}")
print(f"Orders:    {orders_df.count():,}")

customers_df.show(3)
products_df.show(3)
orders_df.show(3)

Customers: 10,000
Products:  500
Orders:    500,000
+-----------+----------+--------+------+---+
|customer_id|      name|  region|  tier|age|
+-----------+----------+--------+------+---+
|          1|customer_1|Gyeonggi|Bronze| 37|
|          2|customer_2|   Daegu|Silver| 28|
|          3|customer_3|Gyeonggi|Bronze| 57|
+-----------+----------+--------+------+---+
only showing top 3 rows

+----------+------------+-----------+------+
|product_id|product_name|   category| price|
+----------+------------+-----------+------+
|         1|   product_1|     Sports|319382|
|         2|   product_2|Electronics| 57085|
|         3|   product_3|  Furniture| 99758|
+----------+------------+-----------+------+
only showing top 3 rows

+--------+-----------+----------+--------+---------+----------+
|order_id|customer_id|product_id|quantity|   status|order_date|
+--------+-----------+----------+--------+---------+----------+
|       1|       2382|       499|       8| returned|2025-10-04|
|       2|  

---
## 2. Spark SQL Deep Dive

### 2.1 Multi-table JOIN

In [3]:
# Order detail: join customers + orders + products
order_detail = spark.sql("""
    SELECT 
        o.order_id,
        c.name AS customer_name,
        c.region,
        c.tier,
        p.product_name,
        p.category,
        p.price,
        o.quantity,
        (p.price * o.quantity) AS total_amount,
        o.status,
        o.order_date
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'completed'
""")

order_detail.show(5)
print(f"Completed orders: {order_detail.count():,}")

+--------+-------------+-------+--------+------------+--------+-----+--------+------------+---------+----------+
|order_id|customer_name| region|    tier|product_name|category|price|quantity|total_amount|   status|order_date|
+--------+-------------+-------+--------+------------+--------+-----+--------+------------+---------+----------+
|  340505|customer_1697|   Jeju|Platinum|  product_26|   Books|32870|       2|       65740|completed|2025-04-01|
|  325370|customer_2214|  Seoul|  Silver|  product_26|   Books|32870|      10|      328700|completed|2025-04-02|
|  410725|customer_2529|Daejeon|Platinum|  product_26|   Books|32870|       4|      131480|completed|2025-11-15|
|  450404|customer_8075|  Busan|    Gold|  product_26|   Books|32870|      10|      328700|completed|2025-09-23|
|   92975|customer_1145|Daejeon|  Bronze|  product_26|   Books|32870|       5|      164350|completed|2025-02-01|
+--------+-------------+-------+--------+------------+--------+-----+--------+------------+-----

### 2.2 CTE (Common Table Expression) & Subqueries

In [4]:
# CTE: Top category by region sales
spark.sql("""
    WITH regional_sales AS (
        SELECT 
            c.region,
            p.category,
            SUM(p.price * o.quantity) AS total_sales,
            COUNT(*) AS order_count
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'completed'
        GROUP BY c.region, p.category
    ),
    ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY region ORDER BY total_sales DESC) AS rank
        FROM regional_sales
    )
    SELECT region, category, total_sales, order_count
    FROM ranked
    WHERE rank <= 3
    ORDER BY region, rank
""").show(30)

+--------+-----------+-----------+-----------+
|  region|   category|total_sales|order_count|
+--------+-----------+-----------+-----------+
|   Busan|Electronics|10589143311|       6905|
|   Busan|     Sports|10350390942|       7035|
|   Busan|      Books|10064652495|       7174|
|   Daegu|Electronics|10567546025|       7001|
|   Daegu|     Sports|10431199028|       7043|
|   Daegu|  Furniture|10119792463|       7111|
| Daejeon|Electronics|11328775335|       7420|
| Daejeon|  Furniture|11003228451|       7593|
| Daejeon|     Sports|10560886807|       7294|
| Gwangju|Electronics|10321513391|       6864|
| Gwangju|     Sports|10089531950|       6939|
| Gwangju|  Furniture| 9830771372|       6982|
|Gyeonggi|Electronics|11187289664|       7349|
|Gyeonggi|     Sports|10742908891|       7395|
|Gyeonggi|  Furniture|10574051564|       7409|
| Incheon|Electronics|10860998473|       7042|
| Incheon|     Sports|10464969454|       7119|
| Incheon|  Furniture|10365603087|       7297|
|    Jeju|Ele

In [5]:
# Correlated subquery: customers who purchased more than average
spark.sql("""
    SELECT 
        c.customer_id,
        c.name,
        c.tier,
        customer_orders.total_spent
    FROM customers c
    JOIN (
        SELECT 
            customer_id,
            SUM(quantity) AS total_quantity,
            COUNT(*) AS order_count,
            SUM(quantity * 1) AS total_spent
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
        HAVING COUNT(*) > (
            SELECT AVG(cnt) FROM (
                SELECT customer_id, COUNT(*) AS cnt
                FROM orders WHERE status = 'completed'
                GROUP BY customer_id
            )
        )
    ) customer_orders ON c.customer_id = customer_orders.customer_id
    ORDER BY customer_orders.total_spent DESC
    LIMIT 10
""").show()

+-----------+-------------+--------+-----------+
|customer_id|         name|    tier|total_spent|
+-----------+-------------+--------+-----------+
|        558| customer_558|  Silver|        329|
|       7441|customer_7441|  Silver|        326|
|       6325|customer_6325|  Bronze|        325|
|       3527|customer_3527|    Gold|        322|
|       5858|customer_5858|    Gold|        322|
|       9422|customer_9422|    Gold|        321|
|       6154|customer_6154|Platinum|        312|
|       4005|customer_4005|Platinum|        312|
|       7067|customer_7067|  Silver|        310|
|       5451|customer_5451|    Gold|        306|
+-----------+-------------+--------+-----------+



### 2.3 Window Functions Deep Dive

In [6]:
# Window functions via DataFrame API
window_region = Window.partitionBy("region").orderBy(F.col("total_spent").desc())
window_region_all = Window.partitionBy("region")

customer_spending = (
    orders_df
    .filter(F.col("status") == "completed")
    .join(customers_df, "customer_id")
    .join(products_df, "product_id")
    .groupBy("customer_id", "name", "region")
    .agg(F.sum(F.col("price") * F.col("quantity")).alias("total_spent"))
)

result = customer_spending.select(
    "customer_id",
    "name",
    "region",
    "total_spent",
    F.rank().over(window_region).alias("region_rank"),
    F.percent_rank().over(window_region).alias("percentile"),
    (F.col("total_spent") / F.sum("total_spent").over(window_region_all) * 100)
        .alias("region_share_pct")
).filter(F.col("region_rank") <= 3)

result.orderBy("region", "region_rank").show(30)

+-----------+-------------+--------+-----------+-----------+--------------------+-------------------+
|customer_id|         name|  region|total_spent|region_rank|          percentile|   region_share_pct|
+-----------+-------------+--------+-----------+-----------+--------------------+-------------------+
|        558| customer_558|   Busan|   93910865|          1|                 0.0| 0.1599430112089668|
|       9995|customer_9995|   Busan|   82033629|          2|8.084074373484236E-4|0.13971445841393565|
|       5102|customer_5102|   Busan|   78115106|          3|0.001616814874696...|0.13304067931381136|
|       2011|customer_2011|   Daegu|   93120594|          1|                 0.0| 0.1588413653772132|
|       3689|customer_3689|   Daegu|   90993802|          2|8.149959250203749E-4| 0.1552135690902465|
|       6409|customer_6409|   Daegu|   82977970|          3|0.001629991850040...|0.14154048513725584|
|       5858|customer_5858| Daejeon|   91951230|          1|                 0.0|0

In [7]:
# Time-series window: cumulative sum, moving average
monthly_sales = spark.sql("""
    SELECT 
        SUBSTRING(order_date, 1, 7) AS month,
        SUM(o.quantity * p.price) AS monthly_sales
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'completed'
        GROUP BY SUBSTRING(order_date, 1, 7)
        ORDER BY month
""")

monthly_sales.createOrReplaceTempView("monthly_sales")

spark.sql("""
    SELECT 
        month,
        monthly_sales,
        SUM(monthly_sales) OVER (ORDER BY month) AS cumulative_sales,
        AVG(monthly_sales) OVER (
            ORDER BY month 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS moving_avg_3m,
        monthly_sales - LAG(monthly_sales, 1) OVER (ORDER BY month) AS mom_change
    FROM monthly_sales
""").show(12)

+-------+-------------+----------------+--------------------+-----------+
|  month|monthly_sales|cumulative_sales|       moving_avg_3m| mom_change|
+-------+-------------+----------------+--------------------+-----------+
|2025-01|  40306408128|     40306408128|     4.0306408128E10|       NULL|
|2025-02|  39090165092|     79396573220|      3.969828661E10|-1216243036|
|2025-03|  39798186722|    119194759942|3.973158664733333...|  708021630|
|2025-04|  39945339358|    159140099300|3.961123039066666...|  147152636|
|2025-05|  39466575518|    198606674818|3.973670053266666...| -478763840|
|2025-06|  40110601960|    238717276778|3.984083894533333...|  644026442|
|2025-07|  39850232478|    278567509256|     3.9809136652E10| -260369482|
|2025-08|  39540502123|    318108011379|3.983377885366666...| -309730355|
|2025-09|  39530800252|    357638811631|3.964051161766666...|   -9701871|
|2025-10|  39610036660|    397248848291|     3.9560446345E10|   79236408|
|2025-11|  39853949782|    43710279807

---
## 3. How the Catalyst Optimizer Works

Catalyst is Spark SQL's query optimization engine.

```
SQL / DataFrame API
        │
        ▼
┌─────────────────────┐
│ Unresolved Logical  │  only column/table names
│ Plan                │
└─────────┬───────────┘
          │  Analysis (resolve names via catalog)
          ▼
┌─────────────────────┐
│ Logical Plan        │  includes type information
└─────────┬───────────┘
          │  Optimization (rule-based)
          ▼
┌─────────────────────┐
│ Optimized Logical   │  Predicate Pushdown, Column Pruning, etc.
│ Plan                │
└─────────┬───────────┘
          │  Physical Planning (cost-based selection)
          ▼
┌─────────────────────┐
│ Physical Plan       │  actual execution strategy (join type, etc.)
└─────────┬───────────┘
          │  Code Generation (Tungsten)
          ▼
┌─────────────────────┐
│ RDDs                │  actual execution
└─────────────────────┘
```

`explain(mode)` options:
- `"simple"` — Physical Plan only
- `"extended"` — Logical + Physical
- `"codegen"` — generated Java code
- `"cost"` — includes cost information
- `"formatted"` — human-readable format

In [8]:
# Trace the full optimization pipeline of a single query
query = (
    orders_df
    .join(customers_df, "customer_id")
    .join(products_df, "product_id")
    .filter(F.col("status") == "completed")
    .filter(F.col("region") == "Seoul")
    .groupBy("category")
    .agg(F.sum(F.col("price") * F.col("quantity")).alias("total"))
    .orderBy(F.col("total").desc())
)

print("=" * 60)
print("FORMATTED Execution Plan")
print("=" * 60)
query.explain("formatted")

FORMATTED Execution Plan
== Physical Plan ==
* Sort (26)
+- Exchange (25)
   +- * HashAggregate (24)
      +- Exchange (23)
         +- * HashAggregate (22)
            +- * Project (21)
               +- * SortMergeJoin Inner (20)
                  :- * Sort (14)
                  :  +- Exchange (13)
                  :     +- * Project (12)
                  :        +- * SortMergeJoin Inner (11)
                  :           :- * Sort (5)
                  :           :  +- Exchange (4)
                  :           :     +- * Project (3)
                  :           :        +- * Filter (2)
                  :           :           +- * Scan ExistingRDD (1)
                  :           +- * Sort (10)
                  :              +- Exchange (9)
                  :                 +- * Project (8)
                  :                    +- * Filter (7)
                  :                       +- * Scan ExistingRDD (6)
                  +- * Sort (19)
                     +- Ex

In [9]:
# extended: full step-by-step plans
print("=" * 60)
print("EXTENDED Execution Plan (Logical → Optimized → Physical)")
print("=" * 60)
query.explain("extended")

EXTENDED Execution Plan (Logical → Optimized → Physical)
== Parsed Logical Plan ==
'Sort ['total DESC NULLS LAST], true
+- Aggregate [category#12], [category#12, sum((price#13L * quantity#21L)) AS total#472L]
   +- Filter (region#2 = Seoul)
      +- Filter (status#22 = completed)
         +- Project [product_id#20L, customer_id#19L, order_id#18L, quantity#21L, status#22, order_date#23, name#1, region#2, tier#3, age#4L, product_name#11, category#12, price#13L]
            +- Join Inner, (product_id#20L = product_id#10L)
               :- Project [customer_id#19L, order_id#18L, product_id#20L, quantity#21L, status#22, order_date#23, name#1, region#2, tier#3, age#4L]
               :  +- Join Inner, (customer_id#19L = customer_id#0L)
               :     :- LogicalRDD [order_id#18L, customer_id#19L, product_id#20L, quantity#21L, status#22, order_date#23], false
               :     +- LogicalRDD [customer_id#0L, name#1, region#2, tier#3, age#4L], false
               +- LogicalRDD [produc

### How to Read Execution Plans

| Keyword | Meaning |
|--------|------|
| **Scan** | Read from data source |
| **Filter** | Conditional filter (WHERE) |
| **Project** | Column selection (SELECT) |
| **BroadcastHashJoin** | Broadcast small table and join |
| **SortMergeJoin** | Sort both sides then merge-join |
| **HashAggregate** | Hash-based aggregation |
| **Exchange** | Shuffle (data movement over the network) |
| **Sort** | Sorting |

**Key insight:** Fewer Exchange (Shuffle) nodes = better!

---
## 4. Optimization Technique Experiments

### 4.1 Predicate Pushdown

Push filter conditions as close to the data source as possible to reduce the amount of data read.

In [10]:
# Query that looks unoptimized: filter AFTER join
late_filter = (
    orders_df
    .join(customers_df, "customer_id")
    .filter(F.col("region") == "Seoul")
    .filter(F.col("status") == "completed")
)

print("=== FILTER after JOIN (Catalyst auto-applies pushdown) ===")
late_filter.explain()

print()

# Manual: filter BEFORE join
early_filter = (
    orders_df.filter(F.col("status") == "completed")
    .join(
        customers_df.filter(F.col("region") == "Seoul"),
        "customer_id"
    )
)

print("=== Manual FILTER before JOIN ===")
early_filter.explain()

print()
print("💡 Check whether the two execution plans are identical!")
print("   Catalyst automatically applies Predicate Pushdown.")

=== FILTER after JOIN (Catalyst auto-applies pushdown) ===
== Physical Plan ==
*(5) Project [customer_id#19L, order_id#18L, product_id#20L, quantity#21L, status#22, order_date#23, name#1, region#2, tier#3, age#4L]
+- *(5) SortMergeJoin [customer_id#19L], [customer_id#0L], Inner
   :- *(2) Sort [customer_id#19L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#19L, 200), ENSURE_REQUIREMENTS, [plan_id=1394]
   :     +- *(1) Filter ((isnotnull(status#22) AND (status#22 = completed)) AND isnotnull(customer_id#19L))
   :        +- *(1) Scan ExistingRDD[order_id#18L,customer_id#19L,product_id#20L,quantity#21L,status#22,order_date#23]
   +- *(4) Sort [customer_id#0L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#0L, 200), ENSURE_REQUIREMENTS, [plan_id=1400]
         +- *(3) Filter ((isnotnull(region#2) AND (region#2 = Seoul)) AND isnotnull(customer_id#0L))
            +- *(3) Scan ExistingRDD[customer_id#0L,name#1,region#2,tier#3,age#4L]



=

### 4.2 Column Pruning

Read only the needed columns to save I/O and memory.

In [11]:
# Fetch all columns but use only a subset
all_cols = (
    orders_df
    .join(customers_df, "customer_id")
    .select("name", "status")
)

print("=== Column Pruning verification ===")
all_cols.explain()

print()
print("💡 In the Scan node of the execution plan, notice that only")
print("   the needed columns [name, customer_id] etc. are read.")
print("   Compared to SELECT *, unnecessary columns are eliminated.")

=== Column Pruning verification ===
== Physical Plan ==
*(5) Project [name#1, status#22]
+- *(5) SortMergeJoin [customer_id#19L], [customer_id#0L], Inner
   :- *(2) Sort [customer_id#19L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#19L, 200), ENSURE_REQUIREMENTS, [plan_id=1516]
   :     +- *(1) Project [customer_id#19L, status#22]
   :        +- *(1) Filter isnotnull(customer_id#19L)
   :           +- *(1) Scan ExistingRDD[order_id#18L,customer_id#19L,product_id#20L,quantity#21L,status#22,order_date#23]
   +- *(4) Sort [customer_id#0L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#0L, 200), ENSURE_REQUIREMENTS, [plan_id=1522]
         +- *(3) Project [customer_id#0L, name#1]
            +- *(3) Filter isnotnull(customer_id#0L)
               +- *(3) Scan ExistingRDD[customer_id#0L,name#1,region#2,tier#3,age#4L]



💡 In the Scan node of the execution plan, notice that only
   the needed columns [name, customer_id] etc. are read.
  

### 4.3 Constant Folding

Pre-compute constant expressions at compile time.

In [12]:
# Query containing constant expressions
constant_query = orders_df.filter(
    F.col("quantity") > F.lit(2) + F.lit(3)  # folded to 5
)

print("=== Constant Folding ===")
constant_query.explain("extended")

print()
print("💡 In the Optimized Logical Plan, verify that (2 + 3) has been folded to 5.")

=== Constant Folding ===
== Parsed Logical Plan ==
'Filter ('quantity > (2 + 3))
+- LogicalRDD [order_id#18L, customer_id#19L, product_id#20L, quantity#21L, status#22, order_date#23], false

== Analyzed Logical Plan ==
order_id: bigint, customer_id: bigint, product_id: bigint, quantity: bigint, status: string, order_date: string
Filter (quantity#21L > cast((2 + 3) as bigint))
+- LogicalRDD [order_id#18L, customer_id#19L, product_id#20L, quantity#21L, status#22, order_date#23], false

== Optimized Logical Plan ==
Filter (isnotnull(quantity#21L) AND (quantity#21L > 5))
+- LogicalRDD [order_id#18L, customer_id#19L, product_id#20L, quantity#21L, status#22, order_date#23], false

== Physical Plan ==
*(1) Filter (isnotnull(quantity#21L) AND (quantity#21L > 5))
+- *(1) Scan ExistingRDD[order_id#18L,customer_id#19L,product_id#20L,quantity#21L,status#22,order_date#23]


💡 In the Optimized Logical Plan, verify that (2 + 3) has been folded to 5.


### 4.4 Controlling Broadcast Join

A small table can be **broadcast** to every executor so the large table never has to be shuffled (`BroadcastHashJoin`). Spark decides automatically by comparing the *estimated size* of the smaller side against `spark.sql.autoBroadcastJoinThreshold` (default 10MB).

**Important catch — auto-broadcast depends on size statistics:**
A DataFrame built with `createDataFrame(python_list)` is backed by an `ExistingRDD`, which has **no size statistics**. Spark therefore assumes it is enormous (`sizeInBytes ≈ Long.MaxValue`) and **never auto-broadcasts it** — you always get a `SortMergeJoin`, no matter how you set the threshold. This is exactly why a naive "broadcast vs sort-merge" benchmark on in-memory data shows no difference: both sides stay `SortMergeJoin`.

Two ways to make it work:
1. **Read from a real source (Parquet)** → Spark gets accurate file statistics → auto-broadcast fires.
2. **`F.broadcast()` hint** → forces a broadcast regardless of statistics.

The cells below demonstrate: (1) the in-memory table failing to auto-broadcast, (2) Parquet fixing it, and (3) the real `SortMergeJoin` vs `BroadcastHashJoin` performance difference once a genuine contrast exists.

In [13]:
# Helpers to inspect the chosen join algorithm and Spark's estimated table size
def join_type(df):
    plan = df._jdf.queryExecution().executedPlan().toString()
    for t in ["BroadcastHashJoin", "SortMergeJoin", "ShuffledHashJoin", "BroadcastNestedLoopJoin"]:
        if t in plan:
            return t
    return "Unknown"

def est_size_bytes(df):
    return int(df._jdf.queryExecution().optimizedPlan().stats().sizeInBytes())

print(f"autoBroadcastJoinThreshold = {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}\n")

# (1) createDataFrame-backed small table -> NO statistics -> assumed huge -> NO auto-broadcast
print("--- (1) In-memory products (createDataFrame / ExistingRDD, no stats) ---")
print(f"estimated size : {est_size_bytes(products_df):,} bytes  (≈ Long.MaxValue = 'unknown, assume huge')")
print(f"join algorithm : {join_type(orders_df.join(products_df, 'product_id'))}   <- auto-broadcast SKIPPED\n")

# (2) Persist to Parquet -> Spark gets real file statistics -> auto-broadcast fires
products_path = "/home/jovyan/data/nb02_products.parquet"
orders_path   = "/home/jovyan/data/nb02_orders.parquet"
products_df.write.mode("overwrite").parquet(products_path)
orders_df.write.mode("overwrite").parquet(orders_path)
products_pq = spark.read.parquet(products_path)
orders_pq   = spark.read.parquet(orders_path)

print("--- (2) Parquet-backed products (real statistics) ---")
print(f"estimated size : {est_size_bytes(products_pq):,} bytes  (< 10MB threshold)")
print(f"join algorithm : {join_type(orders_pq.join(products_pq, 'product_id'))}   <- auto-broadcast FIRES\n")

# (3) Explicit broadcast() hint works regardless of statistics
print("--- (3) Explicit F.broadcast() hint on the in-memory table ---")
print(f"join algorithm : {join_type(orders_df.join(F.broadcast(products_df), 'product_id'))}   <- forced\n")

print("=== Physical plan of the auto-broadcast (Parquet) join ===")
orders_pq.join(products_pq, "product_id").explain()

autoBroadcastJoinThreshold = 10485760b

--- (1) In-memory products (createDataFrame / ExistingRDD, no stats) ---
estimated size : 9,223,372,036,854,775,807 bytes  (≈ Long.MaxValue = 'unknown, assume huge')
join algorithm : SortMergeJoin   <- auto-broadcast SKIPPED

--- (2) Parquet-backed products (real statistics) ---
estimated size : 13,231 bytes  (< 10MB threshold)
join algorithm : BroadcastHashJoin   <- auto-broadcast FIRES

--- (3) Explicit F.broadcast() hint on the in-memory table ---
join algorithm : BroadcastHashJoin   <- forced

=== Physical plan of the auto-broadcast (Parquet) join ===
== Physical Plan ==
*(2) Project [product_id#541L, order_id#539L, customer_id#540L, quantity#542L, status#543, order_date#544, product_name#532, category#533, price#534L]
+- *(2) BroadcastHashJoin [product_id#541L], [product_id#531L], Inner, BuildRight, false
   :- *(2) Filter isnotnull(product_id#541L)
   :  +- *(2) ColumnarToRow
   :     +- FileScan parquet [order_id#539L,customer_id#540L,prod

In [14]:
import time

def best_of(fn, repeat=3):
    """Warm up once (codegen, caches), then return the best of N timed runs."""
    fn()
    times = []
    for _ in range(repeat):
        t0 = time.time(); fn(); times.append(time.time() - t0)
    return min(times)

# Use the Parquet-backed tables so the planner can choose based on REAL size statistics.
# An aggregation over the join forces all rows through, making the shuffle cost visible.
def run_join():
    (orders_pq.join(products_pq, "product_id")
        .groupBy("category")
        .agg(F.sum(F.col("price") * F.col("quantity")).alias("total"))
        .collect())

# --- SortMergeJoin: disable broadcast so BOTH sides are shuffled by product_id ---
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print(f"[broadcast disabled] join algorithm = {join_type(orders_pq.join(products_pq, 'product_id'))}")
smj_time = best_of(run_join)

# --- BroadcastHashJoin: re-enable; small side broadcast, large 'orders' NOT shuffled ---
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
print(f"[broadcast enabled ] join algorithm = {join_type(orders_pq.join(products_pq, 'product_id'))}")
bhj_time = best_of(run_join)

print(f"\nSortMergeJoin     : {smj_time:.3f}s   (shuffles the large 'orders' table)")
print(f"BroadcastHashJoin : {bhj_time:.3f}s   (no large-side shuffle)")
print(f"\n📊 BroadcastHashJoin is {smj_time/bhj_time:.1f}x faster — the Shuffle-elimination effect")
print("   (Now the contrast is real: the two runs use DIFFERENT join algorithms.)")

[broadcast disabled] join algorithm = SortMergeJoin
[broadcast enabled ] join algorithm = BroadcastHashJoin

SortMergeJoin     : 0.458s   (shuffles the large 'orders' table)
BroadcastHashJoin : 0.213s   (no large-side shuffle)

📊 BroadcastHashJoin is 2.2x faster — the Shuffle-elimination effect
   (Now the contrast is real: the two runs use DIFFERENT join algorithms.)


---
## 5. DataFrame API vs SQL: Identical Execution Plans

DataFrame API and SQL both go through the **same Catalyst pipeline**, so equivalent queries produce identical execution plans.

In [15]:
# Via DataFrame API
df_way = (
    orders_df
    .filter(F.col("status") == "completed")
    .join(customers_df, "customer_id")
    .groupBy("region")
    .agg(F.count("*").alias("order_count"))
    .orderBy(F.col("order_count").desc())
)

# Via SQL
sql_way = spark.sql("""
    SELECT c.region, COUNT(*) AS order_count
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.region
    ORDER BY order_count DESC
""")

print("=== DataFrame API Execution Plan ===")
df_way.explain()

print("\n=== SQL Execution Plan ===")
sql_way.explain()

print("\n💡 Verify that the two execution plans are identical!")
print("   Regardless of API used, Catalyst applies the same optimizations.")
print("   → Choose based on team convention and readability.")

=== DataFrame API Execution Plan ===
== Physical Plan ==
*(7) Sort [order_count#819L DESC NULLS LAST], true, 0
+- Exchange rangepartitioning(order_count#819L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2831]
   +- *(6) HashAggregate(keys=[region#2], functions=[count(1)])
      +- Exchange hashpartitioning(region#2, 200), ENSURE_REQUIREMENTS, [plan_id=2827]
         +- *(5) HashAggregate(keys=[region#2], functions=[partial_count(1)])
            +- *(5) Project [region#2]
               +- *(5) SortMergeJoin [customer_id#19L], [customer_id#0L], Inner
                  :- *(2) Sort [customer_id#19L ASC NULLS FIRST], false, 0
                  :  +- Exchange hashpartitioning(customer_id#19L, 200), ENSURE_REQUIREMENTS, [plan_id=2812]
                  :     +- *(1) Project [customer_id#19L]
                  :        +- *(1) Filter ((isnotnull(status#22) AND (status#22 = completed)) AND isnotnull(customer_id#19L))
                  :           +- *(1) Scan ExistingRDD[order_id#18L

---
## 6. Anti-patterns and Correct Patterns

Patterns that Catalyst cannot optimize.

In [16]:
# ❌ Anti-pattern 1: Python UDF cannot be optimized by Catalyst
from pyspark.sql.functions import udf

@udf("boolean")
def is_high_salary_udf(salary):
    return salary > 80000

# ✅ Correct pattern: use built-in functions
print("=== UDF (cannot be optimized) ===")
udf_plan = customers_df.filter(is_high_salary_udf(F.col("age")))  # Catalyst cannot see inside
udf_plan.explain()

print("\n=== Built-in function (optimizable) ===")
builtin_plan = customers_df.filter(F.col("age") > 80000)
builtin_plan.explain()

print("\n💡 UDFs show 'pythonUDF' in the plan.")
print("   Catalyst cannot look inside a UDF, so optimization is limited.")
print("   Prefer pyspark.sql.functions built-ins whenever possible.")

=== UDF (cannot be optimized) ===
== Physical Plan ==
*(2) Project [customer_id#0L, name#1, region#2, tier#3, age#4L]
+- *(2) Filter pythonUDF0#831: boolean
   +- BatchEvalPython [is_high_salary_udf(age#4L)#830], [pythonUDF0#831]
      +- *(1) Scan ExistingRDD[customer_id#0L,name#1,region#2,tier#3,age#4L]



=== Built-in function (optimizable) ===
== Physical Plan ==
*(1) Filter (isnotnull(age#4L) AND (age#4L > 80000))
+- *(1) Scan ExistingRDD[customer_id#0L,name#1,region#2,tier#3,age#4L]



💡 UDFs show 'pythonUDF' in the plan.
   Catalyst cannot look inside a UDF, so optimization is limited.
   Prefer pyspark.sql.functions built-ins whenever possible.


In [17]:
import time

# Performance comparison: UDF vs built-in functions
large_df = spark.range(1_000_000).withColumn("value", (F.col("id") % 100).cast("int"))

@udf("int")
def multiply_udf(x):
    return x * 2 if x is not None else None

# UDF
start = time.time()
large_df.withColumn("doubled", multiply_udf(F.col("value"))).count()
udf_time = time.time() - start

# Built-in function
start = time.time()
large_df.withColumn("doubled", F.col("value") * 2).count()
builtin_time = time.time() - start

print(f"UDF:           {udf_time:.3f}s")
print(f"Built-in func: {builtin_time:.3f}s")
print(f"Built-in is {udf_time/builtin_time:.1f}x faster")
print(f"\n💡 UDF incurs Python ↔ JVM serialization overhead + no Tungsten optimization")

UDF:           0.055s
Built-in func: 0.024s
Built-in is 2.3x faster

💡 UDF incurs Python ↔ JVM serialization overhead + no Tungsten optimization


In [18]:
# ❌ Anti-pattern 2: collect() then process in Python
# BAD: data = orders_df.collect()  # pulls all data to the driver
# BAD: for row in data: ...        # iterates in Python

# ✅ Correct pattern: process with Spark API
print("❌ BAD: df.collect() → Python iteration → risk of memory explosion")
print("✅ GOOD: df.groupBy().agg() → Spark handles distributed processing")
print()
print("❌ BAD: df.toPandas() (large data)")
print("✅ GOOD: df.toPandas() (small results only, e.g. after aggregation)")
print()
print("❌ BAD: rdd.map(lambda) → Python UDF")
print("✅ GOOD: F.when(), F.col(), F.expr() → built-in functions")

❌ BAD: df.collect() → Python iteration → risk of memory explosion
✅ GOOD: df.groupBy().agg() → Spark handles distributed processing

❌ BAD: df.toPandas() (large data)
✅ GOOD: df.toPandas() (small results only, e.g. after aggregation)

❌ BAD: rdd.map(lambda) → Python UDF
✅ GOOD: F.when(), F.col(), F.expr() → built-in functions


---
## 📝 Key Summary

| Concept | Description |
|------|------|
| **Catalyst Optimizer** | Engine that automatically optimizes SQL/DataFrame queries |
| **Predicate Pushdown** | Pushes filters toward the data source to reduce data read |
| **Column Pruning** | Reads only needed columns to reduce I/O |
| **Constant Folding** | Pre-computes constant expressions |
| **Broadcast Join** | Copies small table to all nodes to eliminate Shuffle |
| **explain()** | View execution plan (formatted, extended, cost) |
| **Avoid UDFs** | Python UDFs incur serialization cost + cannot be optimized → prefer built-ins |

### Next Step (Step 3)
- Deep dive into Shuffle & Partitioning
- repartition vs coalesce
- Measuring and optimizing Shuffle cost
- Partition design strategies

In [19]:
spark.stop()
print("SparkSession stopped")

SparkSession stopped
